In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ No GPU')

In [ ]:
import subprocess
subprocess.run('apt-get install -y ffmpeg > /dev/null 2>&1', shell=True)
print('✅ ffmpeg installed')

!pip install openai-whisper -q
print('✅ Whisper installed')

!pip install gtts -q
print('✅ TTS installed')

!pip install transformers sentencepiece sacremoses -q
print('✅ Transformers installed')

!pip install pydub soundfile -q
print('✅ Audio tools installed')

In [ ]:
!pip install gdown -q
!gdown --id 1dYALfQSq7FtvPD-CxYKS0iXCyNWJXe9t -O supernan_source.mp4
print('✅ Video downloaded')

In [ ]:
import os, subprocess
os.makedirs('output', exist_ok=True)

subprocess.run(
    'ffmpeg -y -ss 30 -i supernan_source.mp4 -t 15 '
    '-c:v libx264 -crf 18 -preset fast -c:a aac output/clip.mp4',
    shell=True, check=True, capture_output=True
)
subprocess.run(
    'ffmpeg -y -i output/clip.mp4 -ar 16000 -ac 1 -vn output/clip.wav',
    shell=True, check=True, capture_output=True
)
print('✅ Clip extracted')

In [ ]:
import whisper

model = whisper.load_model('medium')
result = model.transcribe('output/clip.wav', language='en')
english_text = result['text'].strip()
print(f'English: {english_text}')

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

tok = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-en-hi')
mt = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-en-hi')
batch = tok([english_text], return_tensors='pt', padding=True, truncation=True, max_length=512)
out = mt.generate(**batch)
hindi_text = tok.decode(out[0], skip_special_tokens=True)
print(f'Hindi: {hindi_text}')

In [ ]:
import subprocess
from gtts import gTTS
from pydub import AudioSegment

tts = gTTS(text=hindi_text, lang='hi', slow=False)
tts.save('output/hindi_speech.mp3')

subprocess.run(
    'ffmpeg -y -i output/hindi_speech.mp3 -ar 16000 -ac 1 output/hindi_speech.wav',
    shell=True, check=True, capture_output=True
)
print('✅ Hindi audio generated')

speech_dur = len(AudioSegment.from_wav('output/hindi_speech.wav')) / 1000.0
factor = speech_dur / 15.0
print(f'Speech: {speech_dur:.2f}s | Factor: {factor:.2f}x')

subprocess.run(
    f'ffmpeg -y -i output/hindi_speech.wav '
    f'-filter:a "atempo={factor:.4f}" output/hindi_adj.wav',
    shell=True, check=True, capture_output=True
)
print('✅ Audio adjusted')

In [ ]:
import subprocess
from IPython.display import FileLink, Video, display
from google.colab import files

subprocess.run(
    'ffmpeg -y -i output/clip.mp4 -i output/hindi_adj.wav '
    '-map 0:v -map 1:a -c:v copy -shortest '
    'output/final_hindi_dubbed.mp4',
    shell=True, check=True, capture_output=True
)
print('✅ Final video ready!')
display(Video('output/final_hindi_dubbed.mp4', width=600)) #For Kaggle
Video("output/final_hindi_dubbed.mp4", embed=True, width=600) #For Colab